# 1강. 텍스트 → 토큰 → ID → 임베딩 벡터
오늘의 핵심 흐름
raw text
→ tokenization
→ token ids
→ embedding lookup
→ dense vectors

예를 들어 문장 "hello"가 있다고 합시다.

"hello"
→ ["h", "e", "l", "l", "o"]
→ [2, 1, 3, 3, 4]
→ [
    vector for h,
    vector for e,
    vector for l,
    vector for l,
    vector for o
  ]

여기서 중요한 점은 모델은 문자열을 직접 이해하지 않는다는 것입니다. 모델이 받는 것은 숫자 ID이고, 그 ID는 embedding matrix의 row를 선택하는 데 사용됩니다.

## 1-1 Character-level tokenizer 직접 만들기

먼저 BPE, WordPiece 같은 복잡한 tokenizer는 잠시 미룹니다. 가장 단순한 character-level tokenizer부터 직접 만듭니다.

In [1]:
text = "hello embedding model"

# 1. vocabulary 만들기
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

print("vocab:", chars)
print("stoi:", stoi)

# 2. encode: string -> ids
def encode(s):
    return [stoi[ch] for ch in s]

# 3. decode: ids -> string
def decode(ids):
    return "".join([itos[i] for i in ids])

ids = encode(text)
print("encoded:", ids)
print("decoded:", decode(ids))

vocab: [' ', 'b', 'd', 'e', 'g', 'h', 'i', 'l', 'm', 'n', 'o']
stoi: {' ': 0, 'b': 1, 'd': 2, 'e': 3, 'g': 4, 'h': 5, 'i': 6, 'l': 7, 'm': 8, 'n': 9, 'o': 10}
encoded: [5, 3, 7, 7, 10, 0, 3, 8, 1, 3, 2, 2, 6, 9, 4, 0, 8, 10, 2, 3, 7]
decoded: hello embedding model


## 1-2. 임베딩 matrix 직접 만들기

이제 token id를 vector로 바꿉니다.

In [4]:
import numpy as np

vocab_size = len(chars)
embedding_dim = 4

# embedding matrix: vocab_size x embedding_dim
E = np.random.randn(vocab_size, embedding_dim)

print("embedding matrix shape:", E.shape)

# token ids
ids = encode("hello")
print("ids:", ids)

# embedding lookup
vectors = E[ids]
print("vectors shape:", vectors.shape)
print(vectors)

embedding matrix shape: (11, 4)
ids: [5, 3, 7, 7, 10]
vectors shape: (5, 4)
[[-1.20634939 -1.75760955 -1.15203693 -0.18978554]
 [-0.19442178 -1.22920947  0.09021379 -0.55476024]
 [ 0.57422082  0.30933298 -1.52013633 -2.40220866]
 [ 0.57422082  0.30933298 -1.52013633 -2.40220866]
 [ 0.02196111  0.44591254  0.66298187 -1.26169938]]


여기서 E는 embedding matrix입니다.

E.shape = [vocab_size, embedding_dim]

ids = [2, 1, 3, 3, 4]라면 E[ids]는 다음을 의미합니다.

[
  E[2],
  E[1],
  E[3],
  E[3],
  E[4]
]

즉, 임베딩은 처음에는 대단한 의미 벡터가 아닙니다. 그냥 학습 가능한 lookup table입니다. 의미는 학습 과정에서 생깁니다.

## 1-3. PyTorch의 nn.Embedding으로 같은 작업 해보기

PyTorch에서는 위 과정을 nn.Embedding이 처리합니다. 공식 문서에서도 nn.Embedding을 “fixed dictionary와 size를 가진 embedding을 저장하는 simple lookup table”로 설명합니다.

In [5]:
import torch
import torch.nn as nn

vocab_size = len(chars)
embedding_dim = 4

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim
)

ids = torch.tensor(encode("hello"), dtype=torch.long)

vectors = embedding(ids)

print("ids shape:", ids.shape)
print("vectors shape:", vectors.shape)
print(vectors)

ids shape: torch.Size([5])
vectors shape: torch.Size([5, 4])
tensor([[-1.4503, -1.5055, -0.1815, -1.2358],
        [ 0.2359,  0.3505, -0.6754,  0.5966],
        [ 1.0708, -1.5249, -2.1662, -1.0760],
        [ 1.0708, -1.5249, -2.1662, -1.0760],
        [ 0.5595, -1.0523, -0.6964, -0.2910]], grad_fn=<EmbeddingBackward0>)


5개 token 각각을
4차원 vector로 바꿨다

## 1-4. 여기서 LLM과 연결되는 지점

LLM의 입력도 기본적으로 이 구조를 가집니다.

text
→ tokenizer
→ input_ids
→ token embeddings
→ positional embeddings
→ Transformer blocks
→ hidden states
→ logits

Transformer는 recurrence나 convolution 없이 attention mechanism을 중심으로 sequence를 처리하는 구조로 제안되었습니다.

In [6]:
import torch
import torch.nn as nn

text = "llm embedding"

chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):
    return [stoi[ch] for ch in s]

def decode(ids):
    return "".join([itos[i] for i in ids])

vocab_size = len(chars)
embedding_dim = 8

embedding = nn.Embedding(vocab_size, embedding_dim)

ids = torch.tensor(encode(text), dtype=torch.long)
x = embedding(ids)

print("text:", text)
print("vocab:", chars)
print("ids:", ids)
print("embedding output shape:", x.shape)
print("first token vector:", x[0])

text: llm embedding
vocab: [' ', 'b', 'd', 'e', 'g', 'i', 'l', 'm', 'n']
ids: tensor([6, 6, 7, 0, 3, 7, 1, 3, 2, 2, 5, 8, 4])
embedding output shape: torch.Size([13, 8])
first token vector: tensor([ 0.5361,  0.5065,  0.2993, -1.1256, -0.4907,  0.3290, -1.2122, -0.5891],
       grad_fn=<SelectBackward0>)
